#### Цель модели — правильно расставить объекты по порядку внутри определенной группы (запроса).

Главные термины:

1. **Query ID (идентификатор запроса / сессии)** - колонка группирует строки  

2. **Релевантность (Target)** - польза от обьекта для запроса  
    - бинарная (0 некупил / 1 купил )

    - многоклассовая (0, 1, 2, 3) -> всплытие

3. **Метрики качества (NDCG и MAP):**  
    - Обычный Accuracy или MAE здесь не работают.

#### Шаг 1. Симулируем работу поисковой строки интернет-магазина


In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

num_queries = 10000 # Количество поисковых запросов
items_per_query = 10 # Каждому пользователю показываем по 10 товаров
total_rows = num_queries * items_per_query

# 1. Генерируем ID запросов (0, 0, 0... 1, 1, 1... 2, 2, 2...)
query_ids = np.repeat(np.arange(num_queries), items_per_query) 

# 2. Генерируем признаки товаров внутри запроса
prices = np.random.uniform(10, 1000, total_rows)       # Цена товара
ratings = np.random.uniform(1.0, 5.0, total_rows)      # Рейтинг (1-5 звезд)
discounts = np.random.choice([0, 10, 20, 30, 50], total_rows, p=[0.5, 0.2, 0.1, 0.1, 0.1]) # Скидка

# Признак релевантности текста (насколько название совпало с запросом)
text_match_score = np.random.uniform(0, 1, total_rows) 

# 3. Симулируем таргет (бизнес-логику поведения человека)
# Покупают чаще то, где высокая релевантность текста, высокий рейтинг и ниже цена
utility = text_match_score * 5 + ratings * 1.5 + (discounts / 10) - (prices / 300) + np.random.normal(0, 1, total_rows)

# Переводим полезность в оценку от 0 до 3 (0-пролистал, 1-клик, 2-корзина, 3-куплено)
target = pd.qcut(utility, q=4, labels=[0, 1, 2, 3]).astype(int)

# Собираем всё в один датафрейм
df = pd.DataFrame({
    'query_id': query_ids,
    'price': prices,
    'rating': ratings,
    'discount': discounts,
    'text_match': text_match_score,
    'target': target
})

print(f"Размер сгенерированного датасета: {df.shape}")
print(df.head(12)) # Посмотрим на первую сессию целиком

`target` — это то, как человек отреагировал на товар. Наша бизнес-логика   
симулировала  реальное поведение людей (люди любят подешевле, с хорошим  
рейтингом и чтобы товар подходил под запрос):

- **`target = 0` (Строки 0, 1, 2):** Покупатель просто пролистал эти кроссовки.  
Они либо слишком дорогие ($951), либо не подходят под запрос.
- **`target = 1` (Строки 3, 5):** Кроссовки заинтересовали человека, он **кликнул**  
на них и зашел почитать отзывы.
- **`target = 3` (Строки 4, 6):** Полный успех! Покупатель **купил** эти товары.
    - _Почему он их купил?_ Кроссовки из строки 4 идеально подошли под его текстовый  
     запрос (`text_match` = 0.88), а кроссовки из строки 6 стоили копейки ($67)   
     при шикарном качестве (рейтинг 4.34). Логика сработала.

#### Задача модели ранжирования

Сейчас эти товары лежат в случайном порядке. Если маркетплейс покажет их пользователю   
в таком виде, то человеку придется пролистать три неинтересных товара (строки 0, 1, 2),   
чтобы дойти до того, что он реально хочет купить (строка 4). Покупатель может устать и  
уйти к конкурентам.

Задача нашей будущей модели — посмотреть на признаки (price, rating, text_match) и   
пересортировать этот список так, чтобы товары с target = 3 оказались на самых первых  
 строчках (индексы 0 и 1), товары с target = 1 шли следом, а «мусор» с target = 0 улетел   
 в самый конец выдачи.

#### Шаг 2. Валидация и подготовка для Ранжирования

В задачах ранжирования есть жёсткое инженерное правило: **один `query_id`  
 нельзя разрывать между train и test**. Все товары из одного запроса должны  
  целиком уйти либо в обучение, либо в валидацию. 

Кроме того, модели вроде `LightGBMRanker` требуют на вход не просто `X` и `y`,   
а специальный массив **groups** — он указывает, сколько строк содержится в каждом   
запросе. В нашем случае в каждом запросе ровно 10 товаров.

In [ ]:
# Нам нужно отсортировать данные по query_id, это критично для ранжирования!
df = df.sort_values(by='query_id').reset_index(drop=True)

# Разделяем запросы: 8000 на обучение, 2000 на валидацию
train_queries = 8000

X_train = df[df['query_id'] < train_queries].drop(columns=['target'])
y_train = df[df['query_id'] < train_queries]['target']

X_val = df[df['query_id'] >= train_queries].drop(columns=['target'])
y_val = df[df['query_id'] >= train_queries]['target']

# Считаем размер групп (сколько товаров привязано к каждому query_id)
# Для LightGBM нужен одномерный массив с длинами групп
groups_train = X_train.groupby('query_id').size().to_numpy()
groups_val = X_val.groupby('query_id').size().to_numpy()

print(f"Обучение: {X_train.shape[0]} строк, {len(groups_train)} уникальных запросов")
print(f"Валидация: {X_val.shape[0]} строк, {len(groups_val)} уникальных запросов")


#### Варианты моделей для текущей задачи

Вариант 1. LightGBM (Класс: `LGBMRanker`) — Идеальный выбор

- **Как работает (LambdaMART / Pairwise-Listwise):** Он не просто предсказывает числа, он берет пары товаров внутри одного запроса и учится понимать, какой из них должен быть выше другого. Он напрямую оптимизирует метрику **NDCG**.
- **Плюсы:** Работает быстрее всех, потребляет мало памяти, и его архитектура идеально заточена под работу с группами (`group` / `lambdarank`).
- **Для инженера:** Это стандарт индустрии. Если на собеседовании просят быстро поднять ранжирование для терабайта логов — пишут на LightGBM.

Вариант 2. CatBoost (Класс: `CatBoostClassifier` или `CatBoost` со специальным лоссом)

- **Как работает:** У CatBoost есть мощнейшие лосс-функции для ранжирования, такие как `YetiRank` и `StochasticFilter`. Они используют продвинутый Listwise-подход (оптимизируют сразу весь список целиком).
- **Плюсы:** Дает высочайшее качество «из коробки» без долгого тюнинга гиперпараметров.
- **Минусы:** Будет обучаться значительно дольше, чем LightGBM на наших группах. CatBoost уже пробовали в других задачах, здесь важнее пощупать новый синтаксис.

Вариант 3. XGBoost (Класс: `XGBRanker`)

- **Как работает:** Использует Pairwise подход (`objective='rank:pairwise'`). Сравнивает товары попарно.
- **Плюсы:** Очень стабилен, имеет строгую регуляризацию.
- **Минусы:** Настройка параметров групп (аргумент `qid`) в XGBoost исторически сделана чуть менее удобно, чем в LightGBM.


Шаг 3. Обучение LGBMRanker

In [ ]:
# LGBMRanker
import lightgbm as lgb
from sklearn.metrics import ndcg_score

# init специальный класс для ранжирования
model_ranker = lgb.LGBMRanker(
    objective="lambdarank", # Специальный режим для ранжирования (оптимизирует NDCG)
    metric="ndcg",          # Контролируем качество по метрике NDCG
    ndcg_eval_at=[1,3,5],   # Считаем NDCG для топ-1, топ-3 и топ-5 товаров в выдаче
    n_estimators=500,       # Количество деревьев
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

# Для обучения нам НЕ нужен query_id в качестве признака, модель узнает о группах из массива group
X_train_features = X_train.drop(columns=['query_id'])
X_val_features = X_val.drop(columns=['query_id'])

model_ranker.fit(
    X_train_features, y_train,              # Обучающая выборка
    group=groups_train,                     # Передаем размеры групп для трейна
    eval_set=[(X_val_features, y_val)],     # Выборка для валидации (экзамен)
    eval_group=[groups_val],                # Передаем размеры групп для валидации
    callbacks=[lgb.early_stopping(stopping_rounds=30), lgb.log_evaluation(period=50)]
)

print("\nМодель успешно обучена!")

**Метрика NDCG@5 = 0.9675:**  
    Это экстремально высокий показатель (близкий к идеальной единице). В реальном поиске Яндекса или Озона   
    метрика NDCG на уровне 0.75–0.85 считается огромным успехом. Высокий результат получился за счет того,   
    что в синтетическом датасете заложили четкую логику (люди покупают дешевое и качественное), и   
    модель мгновенно её раскусила.

---

Шаг 4. Симуляция работы маркетплейса (Ранжирование на практике)

Этот блок кода делает три вещи:

1. Выделяет товары из первого запроса.
2. Модель предсказывает для каждого товара «скор» (ранг) — чем выше скор, тем выше товар должен стоять в выдаче.
3. Сортирует товары по этому скору и сравнивает, что было «до» и что стало «после».

In [ ]:
from sklearn.metrics import ndcg_score

# Делаем предсказание для всей валидационной выборки
val_preds = model_ranker.predict(X_val_features)

# Для расчета ndcg_score из sklearn нам нужно развернуть данные в матрицы (запросы x товары)
# Так как у нас ровно 10 товаров на запрос, мы можем легко сделать reshape
y_val_matrix = y_val.values.reshape(-1, 10)
val_preds_matrix = val_preds.reshape(-1, 10)

# Считаем средний NDCG@5 по всем запросам
final_ndcg_5 = ndcg_score(y_val_matrix, val_preds_matrix, k=5)

print(f"\n Среднее качество ранжирования (NDCG@5) на валидации: {final_ndcg_5:.4f}")


1. Главное концептуальное отличие

- В отличие от классификации и регрессии, где строки оцениваются **изолированно**, в ранжировании модель оценивает объекты **внутри группы (контекста)**.
- Ей неважно точное значение вероятности или цены. Ей важно **выстроить правильный порядок (относительный приоритет)** элементов от лучшего к худшему.

2. Ключевая терминология (Что спросят на собеседовании)

- **Query ID / Session ID**: Колонка-идентификатор, которая группирует объекты. Модель сравнивает товары только внутри одного и того же `query_id` (например, в рамках одного поискового запроса пользователя).
- **Релевантность (Target)**: Степень полезности объекта. Бывает бинарной (кликнул / не кликнул) или градуированной (0 — пролистал, 1 — клик, 2 — корзина, 3 — покупка).
- **Массив Groups**: Специфический инженерный формат данных. Бустинги (LightGBM/XGBoost) требуют на вход одномерный массив с длинами групп (например, `[10, 10, 10...]`), чтобы понимать, где кончается один запрос и начинается другой.

3. Подходы к ранжированию

- **Pointwise**: Модель предсказывает таргет для каждой строки отдельно (по сути, обычная регрессия или классификация). Плохо работает, если порядок критичен.
- **Pairwise**: Модель берет пары объектов внутри одного запроса и учится минимизировать ошибки их взаимного расположения (кто должен быть выше).
- **Listwise**: Самый продвинутый подход (реализован в `LGBMRanker` через _LambdaMART_ и в `CatBoost` через _YetiRank_). Модель оптимизирует всю поисковую выдачу целиком как единый список.

4. Метрики качества

- Accuracy, F1-score или MAE здесь абсолютно бесполезны.
- Стандарт индустрии — **NDCG (Normalized Discounted Cumulative Gain)**. Она оценивает качество порядка и строго штрафует модель, если самый релевантный товар (например, купленный) вместо 1-го места в выдаче оказался на 5-м или 10-м.